# Trích xuất Text Embeddings cho Đồ án Tốt nghiệp (DATN)

Notebook chạy trên Kaggle (GPU) để:
1. Đọc dữ liệu sản phẩm (`items.parquet`) và gộp các trường văn bản (`title`, `category`, `brand`, `features`, `description`) thành một đoạn văn bản đại diện cho mỗi sản phẩm.
2. Dùng mô hình đa phương thức **Jina CLIP v2** (`jinaai/jina-clip-v2`, text encoder) để trích xuất vector đặc trưng văn bản (1024 chiều) trong **cùng không gian biểu diễn** với `image_embeddings.npy` đã trích xuất ở [`imageembedding.ipynb`](./imageembedding.ipynb).
3. Lưu kết quả ra `text_embeddings.npy` + file metadata theo đúng schema (`index`, `item_id`) mà [`docs/qdrant_vector_db_design.md`](../docs/qdrant_vector_db_design.md) đang chờ để nạp bổ sung vào collection `products` trên Qdrant.

Notebook này được viết lại dựa trên toàn bộ các patch và kỹ thuật đã kiểm chứng thành công ở `imageembedding.ipynb` (vá lỗi tương thích Jina CLIP v2 trên Kaggle, tự động phát hiện và tận dụng toàn bộ GPU khả dụng, checkpoint chống gián đoạn), cộng thêm một tối ưu riêng cho văn bản: **length-bucketing** (sắp xếp theo độ dài trước khi chia batch) để giảm padding thừa do độ dài văn bản sản phẩm chênh lệch rất lớn (có sản phẩm mô tả vài chục ký tự, có sản phẩm hàng nghìn ký tự).

## 1. Cài đặt thư viện

Kaggle có sẵn `transformers` nhưng bản cài sẵn có bug khi nạp Jina CLIP v2 (so sánh nhầm `str` với `int` lúc sort state_dict), nên ghim cứng về bản `5.3.0` đã test chạy ổn ở `imageembedding.ipynb`.

**Không** cài `torch` qua pip ở đây (khác bản nháp trước): Kaggle đã cài sẵn đúng bản `torch` build cho GPU của phiên làm việc; `pip install --upgrade torch` có thể kéo về bản build CPU-only từ PyPI và âm thầm vô hiệu hoá GPU. `pillow` được thêm vào vì Jina CLIP v2 là một model class duy nhất cho cả ảnh lẫn văn bản — `from_pretrained` vẫn khởi tạo đầy đủ nhánh thị giác (EVA-02) dù notebook này chỉ gọi `encode_text`, nên vẫn cần đủ các thư viện thị giác để import không lỗi.

In [ ]:
!pip install -q --upgrade "transformers==5.3.0" polars pyarrow pillow tqdm einops timm

## 2. Import thư viện & kiểm tra GPU

Tự động phát hiện số GPU khả dụng (Kaggle có thể cấp 1 GPU P100 hoặc 2 GPU T4 tùy phiên) thay vì hard-code, giống `imageembedding.ipynb`.

In [ ]:
import os
import numpy as np
import polars as pl
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

import torch
from transformers import AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
n_gpu = torch.cuda.device_count()
print(f"Using device: {device} | Số GPU khả dụng: {n_gpu}")
for i in range(n_gpu):
    print(f"  - cuda:{i}: {torch.cuda.get_device_name(i)}")

## 3. Cấu hình đường dẫn dữ liệu

Dùng **đúng dataset Kaggle mà `imageembedding.ipynb` đã dùng thành công** (`hoho0111/clothing-shoes-and-jewelry`), không phải `/kaggle/input/datn-stream-subset` như bản nháp trước. Hai notebook bắt buộc phải đọc cùng một `items.parquet` (152,086 dòng, cùng thứ tự) — đây chính là file đã đối chiếu khớp 100% với `image_embedding_metadata.parquet` trong [`multimodal_embeddings_report.md`](../docs/multimodal_embeddings_report.md) mục 2.3.C. Đọc nhầm dataset khác (kể cả cùng nội dung nhưng khác thứ tự dòng) sẽ làm lệch point ID giữa vector ảnh và vector văn bản khi nạp vào Qdrant.

Có fallback sang `../data` để test nhanh ở local trước khi submit.

In [ ]:
INPUT_DIR = "/kaggle/input/datasets/hoho0111/clothing-shoes-and-jewelry"
OUTPUT_DIR = "/kaggle/working"

MODEL_NAME = "jinaai/jina-clip-v2"

items_path = os.path.join(INPUT_DIR, "items.parquet")
if not os.path.exists(items_path):
    # fallback để test nhanh ở local
    items_path = "../data/items.parquet"
    INPUT_DIR = "../data"

print(f"Loading items from: {items_path}")

## 4. Đọc dữ liệu sản phẩm

In [ ]:
df_items = pl.read_parquet(items_path)
print(f"Tổng số sản phẩm: {len(df_items)}")
print("Các cột dữ liệu:", df_items.columns)
df_items.head(3)

## 5. Tiền xử lý văn bản

Gộp `title`, `category`, `brand`, `features`, `description` thành một chuỗi đại diện cho mỗi sản phẩm, đúng theo template đã thiết kế ở [`multimodal_embeddings_report.md`](../docs/multimodal_embeddings_report.md) mục 3.1 (bản nháp trước thiếu trường `category`). Trường nào rỗng/null thì bỏ qua hẳn thay vì điền placeholder giả (ví dụ `"Brand: unknown"`) — tránh việc nhiều sản phẩm không liên quan bị kéo lại gần nhau trong không gian vector chỉ vì cùng chia sẻ một placeholder.

Việc cắt ngắn văn bản quá dài không cần xử lý thủ công: `encode_text` của Jina CLIP v2 tự cắt theo `max_length` mặc định (8,192 token, đủ cho toàn bộ `description`/`features` dài nhất trong catalog).

In [ ]:
def prepare_text(row):
    parts = []
    if row["title"]:
        parts.append(f"Title: {row['title']}")
    if row["category"]:
        parts.append(f"Category: {row['category']}")
    if row["brand"]:
        parts.append(f"Brand: {row['brand']}")
    if row["features"]:
        parts.append(f"Features: {row['features']}")
    if row["description"]:
        parts.append(f"Description: {row['description']}")

    text = " | ".join(parts).strip()
    return text if text else "unknown"


Điền chuỗi rỗng cho các cột null rồi áp dụng `prepare_text` cho toàn bộ dataset.

In [ ]:
df_items = df_items.with_columns([
    pl.col("title").fill_null(""),
    pl.col("category").fill_null(""),
    pl.col("brand").fill_null(""),
    pl.col("description").fill_null(""),
    pl.col("features").fill_null(""),
])

items_list = df_items.select(["item_id", "title", "category", "brand", "description", "features"]).to_dicts()
texts_to_encode = [prepare_text(item) for item in tqdm(items_list, desc="Gộp văn bản sản phẩm")]
item_ids_list = df_items["item_id"].to_list()

_lens = [len(t) for t in texts_to_encode]
print(f"Ví dụ văn bản sản phẩm đầu tiên:\n\n{texts_to_encode[0]}")
print(f"\nĐộ dài văn bản (ký tự) - min/mean/max: {min(_lens)}/{sum(_lens) // len(_lens)}/{max(_lens)}")

## 6. Tải mô hình Jina CLIP v2

Bản `transformers` cài sẵn trên Kaggle có 3 lỗi tương thích với Jina CLIP v2, cần vá (monkeypatch) trước khi load model — giống hệt `imageembedding.ipynb`:

1. **`dot_natural_key` so sánh nhầm kiểu dữ liệu** khi sort tên tham số trong state_dict → gây `TypeError: '<' not supported between instances of 'str' and 'int'`.
2. **Thiết bị `meta`**: `transformers` khởi tạo model trên thiết bị "ảo" `meta` trước khi nạp trọng số để load nhanh hơn, nên buffer chưa được tính giá trị thật → phải ép về `cpu`.
3. **Buffer non-persistent bị ghi đè bằng vùng nhớ rác**: sau khi nạp xong, `transformers` vẫn ghi đè các buffer không nằm trong checkpoint (RoPE `freqs_cos/sin`, `inv_freq`...) → phải dựng 1 model tham chiếu rồi copy lại đúng giá trị.

Lỗi thứ 3 nguy hiểm nhất vì không ném exception nào cả — model vẫn chạy nhưng ra vector NaN (hoặc sai lệch âm thầm). Vì vậy cuối phần này luôn có bước "smoke test" để bắt lỗi ngay, trước khi chạy trên toàn bộ dataset.

In [ ]:
def patch_dot_natural_key():
    # Lỗi 1/3: sửa hàm sort dùng khi nạp state_dict, không cho so sánh int với str trực tiếp.
    try:
        import transformers.core_model_loading as _core_model_loading
    except ImportError:
        return False
    if not hasattr(_core_model_loading, "dot_natural_key"):
        return False

    def _safe_dot_natural_key(s):
        return [(0, int(p)) if p.isdigit() else (1, p) for p in s.split(".")]

    _core_model_loading.dot_natural_key = _safe_dot_natural_key
    return True

In [ ]:
import contextlib
import gc
import torch.utils._device as _torch_device_mod
from torch.utils._device import _device_constructors


@contextlib.contextmanager
def meta_init_safe_load():
    # Lỗi 2/3: ép mọi tensor tạo trong lúc khởi tạo model về "cpu" thay vì "meta",
    # để buffer được tính giá trị thật ngay từ đầu.
    orig_call = _torch_device_mod.DeviceContext.__torch_function__

    def patched_call(self, func, types, args=(), kwargs=None):
        kwargs = kwargs or {}
        if self.device.type == "meta" and kwargs.get("device") is None and func in _device_constructors():
            kwargs = dict(kwargs)
            kwargs["device"] = "cpu"
            return func(*args, **kwargs)
        return orig_call(self, func, types, args, kwargs)

    _torch_device_mod.DeviceContext.__torch_function__ = patched_call
    try:
        yield
    finally:
        _torch_device_mod.DeviceContext.__torch_function__ = orig_call

In [ ]:
def _iter_non_persistent_buffers(module, prefix=""):
    for name, buf in module._buffers.items():
        if buf is not None and name in module._non_persistent_buffers_set:
            yield (f"{prefix}.{name}" if prefix else name), buf
    for child_name, child in module.named_children():
        child_prefix = f"{prefix}.{child_name}" if prefix else child_name
        yield from _iter_non_persistent_buffers(child, child_prefix)


def restore_non_persistent_buffers(model):
    # Lỗi 3/3 (quan trọng nhất, đã kiểm chứng thực nghiệm): sau from_pretrained,
    # transformers vẫn ghi đè MỌI buffer non-persistent bằng torch.empty_like()
    # (vùng nhớ rác), bất kể patch ở trên đã tính đúng hay chưa.
    # Cách vá: dựng 1 model tham chiếu bằng constructor thẳng (bỏ qua from_pretrained
    # nên không bị ghi đè), rồi copy buffer từ đó sang model thật.
    targets = list(_iter_non_persistent_buffers(model))
    if not targets:
        return []

    with meta_init_safe_load():
        ref_model = type(model)(model.config)
    ref_lookup = dict(_iter_non_persistent_buffers(ref_model))

    restored = []
    for name, buf in targets:
        ref_buf = ref_lookup.get(name)
        if ref_buf is not None and ref_buf.shape == buf.shape:
            buf.data.copy_(ref_buf.data.to(device=buf.device, dtype=buf.dtype))
            restored.append(name)

    del ref_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    missing = [n for n, _ in targets if n not in restored]
    if missing:
        raise RuntimeError(f"Không khôi phục được {len(missing)} buffer: {missing[:10]}")
    return restored

Trước khi load model, cần vá thêm 2 lỗi môi trường trên Kaggle (không liên quan trực tiếp đến việc encode văn bản, nhưng vẫn chặn `import torchvision`/`timm` vì `AutoModel.from_pretrained` luôn khởi tạo cả nhánh thị giác EVA-02 của Jina CLIP v2, kể cả khi ta chỉ định gọi `encode_text`): thiếu thuộc tính `PIL._typing._Ink`, và trùng đăng ký kernel `register_fake` cho `nms`/`roi_align`. Vá thẳng trong RAM (không cần khởi động lại kernel).

In [ ]:
import sys
import os
import types
import typing
import PIL

# 1. Định nghĩa _Ink (Type Hint cho màu sắc) mà một số bản Pillow mới thiếu
InkType = typing.Union[tuple, list, str, int, float, typing.Any]

# 2. Ghi bổ sung vào file vật lý trên ổ đĩa để các lần import sau không bị lỗi
try:
    pil_dir = os.path.dirname(PIL.__file__)
    typing_file = os.path.join(pil_dir, "_typing.py")
    if os.path.exists(typing_file):
        with open(typing_file, "r", encoding="utf-8") as f:
            content = f.read()
        if "_Ink" not in content:
            with open(typing_file, "a", encoding="utf-8") as f:
                f.write("\nimport typing\n_Ink = typing.Any\n")
            print(f"Đã bổ sung _Ink vào file: {typing_file}")
except Exception as e:
    print(f"Bỏ qua ghi file (không bắt buộc): {e}")

# 3. Patch trực tiếp vào RAM hiện tại
try:
    import PIL._typing
    PIL._typing._Ink = InkType
except Exception:
    pass

if "PIL._typing" in sys.modules:
    sys.modules["PIL._typing"]._Ink = InkType
else:
    mod = types.ModuleType("PIL._typing")
    mod._Ink = InkType
    sys.modules["PIL._typing"] = mod

# 4. Xóa cache các module import bị lỗi dở dang để nạp lại sạch sẽ
# (chỉ xóa cache import, không ảnh hưởng biến đang có trong RAM)
modules_to_clear = ["PIL.ImageText", "PIL.ImageDraw", "torchvision", "timm"]
for mod_name in list(sys.modules.keys()):
    if any(mod_name == pkg or mod_name.startswith(pkg + ".") for pkg in modules_to_clear):
        del sys.modules[mod_name]

print("Đã vá xong PIL._typing - không cần khởi động lại kernel.")

In [ ]:
import sys
import torch
import torch.library

# 1. Patch FakeImpl.register (xử lý lỗi register_fake của torchvision::nms)
try:
    import torch._library.fake_impl
    if hasattr(torch._library.fake_impl, "FakeImpl"):
        orig_fake_register = torch._library.fake_impl.FakeImpl.register
        def _safe_fake_register(self, func, *args, **kwargs):
            kwargs["allow_override"] = True
            try:
                return orig_fake_register(self, func, *args, **kwargs)
            except RuntimeError as e:
                if "already has" in str(e).lower() or "fake impl registered" in str(e).lower():
                    return None
                raise e
        torch._library.fake_impl.FakeImpl.register = _safe_fake_register
except Exception as e:
    print(f"Lưu ý FakeImpl: {e}")

# 2. Patch torch.library.register_fake decorator
if hasattr(torch.library, "register_fake"):
    orig_register_fake = torch.library.register_fake
    def _safe_register_fake(op_name, fn=None, *args, **kwargs):
        kwargs["allow_override"] = True
        dec = orig_register_fake(op_name, *args, **kwargs)
        if fn is not None:
            try:
                return dec(fn)
            except RuntimeError as e:
                if "already has" in str(e).lower():
                    return fn
                raise e
        def wrapper(func):
            try:
                return dec(func)
            except RuntimeError as e:
                if "already has" in str(e).lower():
                    return func
                raise e
        return wrapper
    torch.library.register_fake = _safe_register_fake

# 3. Patch torch.library.Library.impl (xử lý lỗi roi_align trước đó)
def patch_torch_library():
    targets = [
        getattr(torch.library, "Library", None),
        getattr(torch.library, "_ScopedLibrary", None),
    ]
    for target in targets:
        if target is not None and hasattr(target, "impl"):
            orig_impl = target.impl
            def make_patched(old_impl):
                def patched_impl(self, *args, **kwargs):
                    kwargs["allow_override"] = True
                    try:
                        return old_impl(self, *args, **kwargs)
                    except Exception as e:
                        err_str = str(e).lower()
                        if "already a kernel registered" in err_str or "already has" in err_str:
                            return None
                        if "allow_override" in err_str:
                            kwargs.pop("allow_override", None)
                            try:
                                return old_impl(self, *args, **kwargs)
                            except Exception as inner_e:
                                if "already a kernel registered" in str(inner_e).lower():
                                    return None
                                raise inner_e
                        raise e
                return patched_impl
            target.impl = make_patched(orig_impl)

patch_torch_library()

# 4. Xóa cache dở dang của torchvision & timm để nạp lại sạch sẽ
for mod_name in list(sys.modules.keys()):
    if any(mod_name == pkg or mod_name.startswith(pkg + ".") for pkg in ["torchvision", "timm"]):
        del sys.modules[mod_name]

# 5. Nạp trực tiếp torchvision và timm
import torchvision
import timm

print("Đã nạp lại torchvision và timm thành công.")

Áp dụng các patch, load model, và khôi phục buffer.

In [ ]:
_patched = patch_dot_natural_key()
print(f"patch_dot_natural_key applied: {_patched}")

print(f"Đang tải mô hình: {MODEL_NAME}...")
with meta_init_safe_load():
    model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True, torch_dtype=torch.float32)
model = model.to(device)
model.eval()
print("Đã tải mô hình thành công.")

_restored = restore_non_persistent_buffers(model)
print(f"Đã khôi phục {len(_restored)} buffer non-persistent (RoPE, ...).")

**Smoke test:** kiểm tra nhanh 1 câu mẫu để chắc chắn model không sinh NaN, trước khi tốn thời gian chạy trên toàn bộ dataset. Không truyền `task=` (mặc định `retrieval.document`, đúng vì đây là văn bản mô tả sản phẩm trong catalog cần lập chỉ mục — `task="retrieval.query"` chỉ dùng lúc encode câu truy vấn của người dùng ở giai đoạn suy luận sau này).

In [ ]:
with torch.no_grad():
    _smoke_emb = model.encode_text(["smoke test"], convert_to_numpy=True, show_progress_bar=False)
_smoke_emb = np.asarray(_smoke_emb, dtype=np.float32)
assert not np.isnan(_smoke_emb).any(), "Model sinh vector NaN ngay ở bước kiểm tra nhanh - dừng lại, không chạy full dataset!"
assert _smoke_emb.shape[-1] == 1024, f"Số chiều vector không khớp (mong đợi 1024, nhận {_smoke_emb.shape[-1]}) - phải cùng không gian với image_embeddings.npy!"
print(f"Smoke test OK - vector mẫu không chứa NaN (shape={_smoke_emb.shape}, dtype={_smoke_emb.dtype}).")

## 7. Trích xuất đặc trưng cho toàn bộ dataset

Hai điểm khác biệt so với `imageembedding.ipynb` (dữ liệu ảnh cố định 224x224, không cần các kỹ thuật này):

- **Length-bucketing:** sắp xếp toàn bộ văn bản theo độ dài trước khi chia batch, để các sản phẩm có độ dài gần nhau nằm cùng batch — giảm token padding thừa do văn bản sản phẩm chênh lệch độ dài rất lớn, tăng thông lượng đáng kể so với batch theo thứ tự ngẫu nhiên gốc. Vì thứ tự xử lý bị xáo trộn, bước gộp kết quả cuối cùng sẽ khớp lại đúng thứ tự gốc của `items.parquet` theo `item_id`.
- **Chia việc cho từng GPU theo kiểu round-robin** (`sorted[gpu_id::n_gpu]`) trên mảng đã sắp xếp thay vì chia thành khối liền kề: mỗi GPU nhận đều một hỗn hợp văn bản ngắn/dài (thay vì 1 GPU chỉ toàn văn bản dài nhất, GPU kia toàn văn bản ngắn nhất), giữ thời gian chạy cân bằng giữa các thiết bị trong khi từng batch nội bộ của mỗi GPU vẫn gần đồng đều độ dài.

Vẫn giữ nguyên cơ chế checkpoint nguyên tử (ghi `.tmp` rồi `os.replace`) và tự động tiếp tục nếu phiên Kaggle bị ngắt giữa chừng, giống `imageembedding.ipynb`.

In [ ]:
import copy
import gc

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints_text")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BATCH_SIZE = 64            # văn bản dài (description) tốn bộ nhớ hơn ảnh cố định 224x224 -> giữ batch vừa phải cho an toàn VRAM
SAVE_EVERY_BATCHES = 20    # ghi checkpoint định kỳ để lỡ notebook bị ngắt cũng không mất hết tiến độ

DEVICES = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
print(f"Chạy song song trên {len(DEVICES)} thiết bị: {DEVICES}")

In [ ]:
n_total = len(texts_to_encode)
sort_order = sorted(range(n_total), key=lambda i: len(texts_to_encode[i]))
sorted_texts = [texts_to_encode[i] for i in sort_order]
sorted_item_ids = [item_ids_list[i] for i in sort_order]

n_dev = len(DEVICES)
text_chunks = [sorted_texts[i::n_dev] for i in range(n_dev)]
id_chunks = [sorted_item_ids[i::n_dev] for i in range(n_dev)]
print(f"Tổng {n_total} sản phẩm, chia thành {[len(c) for c in text_chunks]} sản phẩm/thiết bị (round-robin trên mảng đã sắp xếp theo độ dài).")

**Nhân bản model & sanity check:** nhân bản model sang từng GPU (nếu có nhiều hơn 1) để suy luận song song, rồi kiểm tra nhanh từng bản sao trước khi chạy full dataset.

In [ ]:
print("Phân bổ model sang từng thiết bị...")
model.to(DEVICES[0])
device_models = [model]
for dev in DEVICES[1:]:
    device_models.append(copy.deepcopy(model).to(dev).eval())
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Sanity check: chống vector NaN / vector 0 trên từng bản sao trước khi chạy full dataset
_test_texts = ["a red cotton t-shirt for men", "leather ankle boots with zipper"]
for dev, worker_model in zip(DEVICES, device_models):
    with torch.no_grad():
        emb = worker_model.encode_text(_test_texts, convert_to_numpy=True, show_progress_bar=False)
    emb = np.asarray(emb, dtype=np.float32)
    assert not np.isnan(emb).any(), f"[{dev}] Model sinh vector NaN!"
    assert (np.linalg.norm(emb, axis=1) > 1e-3).all(), f"[{dev}] Vector bị triệt tiêu về 0!"
    print(f"  - {dev}: OK (norm={np.round(np.linalg.norm(emb, axis=1), 4)})")

print("Sanity check thành công trên toàn bộ thiết bị.")

**Hàm xử lý cho từng thiết bị:** văn bản đã nằm sẵn trong RAM (không cần `DataLoader`/nhiều worker đọc đĩa như ảnh), nên chỉ cần cắt batch tuần tự theo `BATCH_SIZE` và suy luận dưới `torch.autocast(float16)` cho tốc độ. Kiểm tra NaN ngay trên từng batch, và ghi checkpoint an toàn (ghi ra file tạm rồi đổi tên bằng `os.replace`). Nếu tìm thấy checkpoint cũ của thiết bị này, tự động bỏ qua phần đã xong và chạy tiếp.

In [ ]:
def existing_progress(gpu_id):
    files = sorted(f for f in os.listdir(CHECKPOINT_DIR) if f.startswith(f"chunk_gpu{gpu_id}_") and f.endswith(".npz"))
    n_items = 0
    for f in files:
        with np.load(os.path.join(CHECKPOINT_DIR, f)) as data:
            n_items += len(data["item_ids"])
    return len(files), n_items


def save_chunk_atomic(gpu_id, chunk_idx, embeddings, item_ids):
    final_path = os.path.join(CHECKPOINT_DIR, f"chunk_gpu{gpu_id}_{chunk_idx:04d}.npz")
    tmp_path = final_path + ".tmp"
    with open(tmp_path, "wb") as fh:
        np.savez_compressed(fh, embeddings=embeddings, item_ids=np.array(item_ids))
    os.replace(tmp_path, final_path)  # ghi nguyên tử -> không bao giờ có file .npz dở dang


def process_device_chunk(gpu_id, device_str, worker_model, chunk_texts, chunk_ids):
    n_done_files, n_done_items = existing_progress(gpu_id)
    if n_done_items > 0:
        print(f"[{device_str}] Tìm thấy checkpoint cũ: {n_done_items} sản phẩm đã xong, tiếp tục từ đó...")
        chunk_texts = chunk_texts[n_done_items:]
        chunk_ids = chunk_ids[n_done_items:]

    chunk_idx = n_done_files
    buf_embs, buf_ids = [], []

    def flush():
        nonlocal chunk_idx
        if not buf_embs:
            return
        save_chunk_atomic(gpu_id, chunk_idx, np.vstack(buf_embs).astype(np.float32), buf_ids)
        buf_embs.clear()
        buf_ids.clear()
        chunk_idx += 1

    n_batches = (len(chunk_texts) + BATCH_SIZE - 1) // BATCH_SIZE
    with torch.no_grad():
        for batch_idx in tqdm(range(n_batches), desc=device_str, position=gpu_id, leave=True):
            start = batch_idx * BATCH_SIZE
            batch_texts = chunk_texts[start:start + BATCH_SIZE]
            batch_ids = chunk_ids[start:start + BATCH_SIZE]

            features = worker_model.encode_text(batch_texts, convert_to_numpy=True, show_progress_bar=False)
            features = np.asarray(features, dtype=np.float32)

            nan_mask = np.isnan(features).any(axis=1)
            if nan_mask.any():
                bad_ids = [batch_ids[j] for j in np.where(nan_mask)[0]]
                raise RuntimeError(f"[{device_str}] Phát hiện vector NaN tại item_id: {bad_ids[:5]}")

            buf_embs.append(features)
            buf_ids.extend(batch_ids)
            if (batch_idx + 1) % SAVE_EVERY_BATCHES == 0:
                flush()
        flush()

    return chunk_idx

**Chạy song song và gộp kết quả:** mỗi thiết bị chạy trên 1 luồng điều phối (suy luận GPU tự giải phóng GIL nên không tranh chấp lẫn nhau), sau đó gộp toàn bộ checkpoint lại. Vì dữ liệu đã bị xáo trộn thứ tự lúc sắp xếp theo độ dài (length-bucketing) và chia round-robin cho các thiết bị, bước gộp phải khớp lại đúng thứ tự gốc của `items.parquet` bằng `item_id` — khác với `imageembedding.ipynb`, nơi thứ tự concatenation theo thiết bị đã trùng khớp thứ tự gốc sẵn nên không cần bước này.

In [ ]:
print(f"Bắt đầu trích xuất {n_total} sản phẩm trên {len(DEVICES)} thiết bị (checkpoint tại {CHECKPOINT_DIR})...")

with ThreadPoolExecutor(max_workers=len(DEVICES)) as executor:
    futures = [
        executor.submit(process_device_chunk, gpu_id, dev, worker_model, chunk_texts, chunk_ids)
        for gpu_id, (dev, worker_model, chunk_texts, chunk_ids) in enumerate(zip(DEVICES, device_models, text_chunks, id_chunks))
    ]
    for f in futures:
        f.result()

print("Đã trích xuất xong, đang gộp checkpoint từ ổ đĩa...")


def load_device_chunks(gpu_id):
    files = sorted(f for f in os.listdir(CHECKPOINT_DIR) if f.startswith(f"chunk_gpu{gpu_id}_") and f.endswith(".npz"))
    embs, ids = [], []
    for f in files:
        with np.load(os.path.join(CHECKPOINT_DIR, f)) as data:
            embs.append(data["embeddings"])
            ids.extend(data["item_ids"].tolist())
    return np.vstack(embs), ids


all_embs, all_ids = [], []
for gpu_id in range(len(DEVICES)):
    embs, ids = load_device_chunks(gpu_id)
    all_embs.append(embs)
    all_ids.extend(ids)

sorted_embeddings = np.vstack(all_embs).astype(np.float32)
assert len(all_ids) == n_total, f"Số lượng ({len(all_ids)}) không khớp tổng ({n_total})!"

# Khớp lại đúng thứ tự gốc của items.parquet theo item_id (xem giải thích ở trên).
id_to_pos = {iid: pos for pos, iid in enumerate(all_ids)}
assert len(id_to_pos) == n_total, "Phát hiện item_id trùng lặp khi gộp checkpoint!"
reorder = np.array([id_to_pos[iid] for iid in item_ids_list])
text_embeddings = sorted_embeddings[reorder]
item_ids_ordered = item_ids_list

print(f"Hoàn tất! Kích thước ma trận embeddings văn bản: {text_embeddings.shape}, dtype: {text_embeddings.dtype}")

## 8. Lưu kết quả

Kiểm tra lại NaN/Inf/dtype lần cuối, sau đó ghi ra file `.tmp` rồi đổi tên (`os.replace`) để không bao giờ để lại file `.npy`/`.parquet` dở dang nếu notebook bị ngắt giữa chừng — giống `imageembedding.ipynb` (bản nháp trước ghi thẳng, có rủi ro để lại file dở dang nếu bị ngắt đúng lúc ghi).

In [ ]:
assert not np.isnan(text_embeddings).any(), "Phát hiện NaN trong embeddings trước khi lưu!"
assert not np.isinf(text_embeddings).any(), "Phát hiện Inf trong embeddings trước khi lưu!"
assert text_embeddings.dtype == np.float32, f"Kiểu dữ liệu không mong đợi: {text_embeddings.dtype}"
assert (item_ids_list == df_items["item_id"].to_list()), "Thứ tự item_id đã bị lệch so với items.parquet!"

emb_output_path = os.path.join(OUTPUT_DIR, "text_embeddings.npy")
with open(emb_output_path + ".tmp", "wb") as fh:
    np.save(fh, text_embeddings)
os.replace(emb_output_path + ".tmp", emb_output_path)  # ghi nguyên tử
print(f"Đã lưu ma trận vector nhúng văn bản tại: {emb_output_path}")

df_metadata = pl.DataFrame({
    "index": list(range(len(item_ids_ordered))),
    "item_id": item_ids_ordered,
})
meta_output_path = os.path.join(OUTPUT_DIR, "text_embedding_metadata.parquet")
df_metadata.write_parquet(meta_output_path + ".tmp")
os.replace(meta_output_path + ".tmp", meta_output_path)  # ghi nguyên tử
print(f"Đã lưu metadata văn bản tại: {meta_output_path}")

## 9. Kiểm tra lại file đã lưu

In [ ]:
loaded_embeddings = np.load(emb_output_path)
print(f"Kiểm tra kích thước file tải lại: {loaded_embeddings.shape}, dtype: {loaded_embeddings.dtype}")
assert not np.isnan(loaded_embeddings).any(), "File đã lưu chứa NaN!"
assert np.allclose(text_embeddings, loaded_embeddings), "Dữ liệu lưu bị lỗi!"
print("Kiểm tra hoàn tất: không có NaN, dữ liệu khớp với bản gốc trong bộ nhớ.")

## 10. Dọn dẹp checkpoint tạm

Sau khi đã xác nhận `text_embeddings.npy` hợp lệ ở bước trên, xoá thư mục checkpoint để không tốn thêm dung lượng `/kaggle/working` (giới hạn output của Kaggle).

In [ ]:
import shutil

if os.path.exists(CHECKPOINT_DIR):
    shutil.rmtree(CHECKPOINT_DIR)
    print(f"Đã xoá checkpoint tạm tại: {CHECKPOINT_DIR}")